# Sesión 9 · Notebook de clase · Transformar columnas y combinar tablas

**Fundamentos de Ciencia de Datos e IA (CDIAG1003)** · Semana 5 · Unidad 3: tema 3.2 (`map`/`apply`, `cut`/`qcut`, `get_dummies`, `merge`, `concat`, formato largo y ancho).

En la sesión 8 cada pregunta de negocio se convirtió en un `groupby`. Hoy se resuelve lo que pasa **antes** de agrupar: la columna que hace falta no existe todavía (hay que **derivarla** de otra), la variable es numérica y la pregunta pide **categorías**, o la información está **repartida en varias tablas** (un archivo por mes, un catálogo de productos, una tabla de locales) y hay que juntarla.

Datos de hoy, en la carpeta `datos/`:

| Archivo | Qué es | De dónde viene |
| :-- | :-- | :-- |
| `ventas_limpio.csv` | las 200 ventas de enero a marzo, limpias | lo que dejó la sesión 7 |
| `tienda.xlsx` | hoja `productos` (costo de cada producto) y hoja `locales` (empleados por ciudad) | sesión 5 |
| `ventas_abril.csv` | 60 ventas de abril, **sin limpiar** | «Su turno 8.1» de la sesión 7 |

Al final del notebook queda un solo archivo, `datos/ventas_completo.csv`: 260 ventas de enero a abril con costo y margen. Es la tabla de la sesión 10.

**Cómo trabajar con este notebook**
- Ejecute las celdas en orden. En los ejercicios **Su turno**, reemplace los `...`; la celda `assert` siguiente comprueba el resultado.
- Los ejercicios no modifican la tabla `v`: trabajan sobre variables nuevas. Después de cada uno hay una celda «Para continuar»: ejecútela siempre.
- Las salidas de la versión resuelta están en pandas 3 (texto con tipo `str`; en pandas 2 aparece `object`).

In [2]:
import numpy as np
import pandas as pd

v = pd.read_csv("datos/ventas_limpio.csv", parse_dates=["fecha"])
print(v.shape, "| ingreso total:", round(v["ingreso"].sum(), 1))
v.head()

(200, 9) | ingreso total: 7424.8


,fecha,producto,categoria,cantidad,precio_unitario,ciudad,metodo_pago,tamano,ingreso
0,2026-01-02,Sartén,Hogar,1,18.0,Quito,tarjeta,pequeña,18.0
1,2026-01-02,Mouse,Electrónica,1,15.0,Machala,efectivo,pequeña,15.0
2,2026-01-03,Galletas,Alimentos,2,1.8,Quito,tarjeta,pequeña,3.6
3,2026-01-03,Cuaderno,Papelería,2,2.5,Guayaquil,transferencia,pequeña,5.0
4,2026-01-04,Cargador,Electrónica,3,12.5,Guayaquil,tarjeta,mediana,37.5


In [3]:
v.dtypes

fecha              datetime64[us]
producto                      str
categoria                     str
cantidad                    int64
precio_unitario           float64
ciudad                        str
metodo_pago                   str
tamano                        str
ingreso                   float64
dtype: object

## 1. Transformar una columna: vectorizado, `map`, `replace`, `apply`

La regla de la sesión 4 sigue vigente: si la operación se puede escribir **columna contra columna** (vectorizada), esa es la forma. Lo nuevo son tres herramientas para cuando la transformación es una **tabla de equivalencias** o una **función**:

| Herramienta | Qué hace | Cuándo |
| :-- | :-- | :-- |
| `serie.map(diccionario)` | sustituye cada valor por su equivalente; lo que no está en el diccionario queda `NaN` | recodificar: ciudad → región, código → nombre |
| `serie.replace({viejo: nuevo})` | sustituye solo los valores indicados; el resto no cambia | corregir unos pocos valores |
| `serie.map(funcion)` / `serie.apply(funcion)` | aplica una función **elemento a elemento** | cálculo que no es vectorizable |
| `df.apply(funcion, axis=1)` | aplica una función **fila a fila** (recibe cada fila como Series) | la regla usa varias columnas |

`apply` recorre la tabla fila por fila en Python: es el plan B, no la primera opción. Con 200 filas no se nota; con un millón, sí.

In [4]:
# Vectorizado: columna contra columna (lo de siempre)
v["ingreso"].equals(v["cantidad"] * v["precio_unitario"])

False

In [ ]:
# map con un diccionario: ciudad -> región. Lo que no está en el diccionario queda NaN.
REGION = {"Guayaquil": "Costa", "Manta": "Costa", "Machala": "Costa", "Quito": "Sierra", "Cuenca": "Sierra"}
v["ciudad"].map(REGION).value_counts(dropna=False)
# v["region"] = v["ciudad"].map(REGION).value_counts(dropna=False)

,fecha,producto,categoria,cantidad,precio_unitario,ciudad,metodo_pago,tamano,ingreso
36,2026-01-19,Mouse,Electrónica,2,15.0,Sin dato,tarjeta,pequeña,30.0
46,2026-01-23,Bolígrafo,Papelería,2,0.6,Sin dato,efectivo,pequeña,1.2
70,2026-02-05,Cargador,Electrónica,1,12.5,Sin dato,efectivo,pequeña,12.5
78,2026-02-10,Chocolate,Alimentos,2,3.2,Sin dato,efectivo,pequeña,6.4
86,2026-02-13,Mouse,Electrónica,1,15.0,Sin dato,efectivo,pequeña,15.0
120,2026-02-28,Chocolate,Alimentos,3,3.2,Sin dato,efectivo,mediana,9.6


In [13]:
# replace cambia solo lo que se indica; lo demás queda igual
v["metodo_pago"].replace({"transferencia": "transf."}).value_counts()


metodo_pago
efectivo       91
tarjeta        73
transf.        27
desconocido     9
Name: count, dtype: int64

In [9]:
# apply fila a fila: una regla que usa dos columnas...
def descuento(fila):
    if fila["cantidad"] >= 5 and fila["metodo_pago"] == "efectivo":
        return 0.10
    return 0.0

con_apply = v.apply(descuento, axis=1)

# ...y la misma regla vectorizada, con np.where (preferible)
vectorizado = np.where((v["cantidad"] >= 5) & (v["metodo_pago"] == "efectivo"), 0.10, 0.0)
print("coinciden:", (con_apply.to_numpy() == vectorizado).all(), "| ventas con descuento:", int((vectorizado > 0).sum()))

coinciden: True | ventas con descuento: 26


### Su turno 1.1 · 4 min
Cree la Series `region` con `map` y el diccionario `REGION`, de modo que las ventas con ciudad «Sin dato» queden como `"Sin dato"` (y no como `NaN`). Cuente las ventas por región con `value_counts()`.

In [22]:
# TODO: reemplace los ... por su código
region = v["ciudad"].map(REGION).fillna("Sin dato")
region.value_counts()

ciudad
Costa       102
Sierra       92
Sin dato      6
Name: count, dtype: int64

In [34]:
assert region.isna().sum() == 0, "las ciudades que no están en el diccionario quedaron NaN: use fillna"
assert region.value_counts().to_dict() == {'Costa': 102, 'Sierra': 92, 'Sin dato': 6}, "revise el diccionario REGION"
print("Correcto: Costa", 102, "ventas, Sierra", 92, "y 'Sin dato'", 6, ".")

Correcto: Costa 102 ventas, Sierra 92 y 'Sin dato' 6 .


In [35]:
# Para continuar con la misma tabla (ejecute siempre esta celda)
v["region"] = v["ciudad"].map(REGION).fillna("Sin dato")

## 2. Categorizar números: `cut` y `qcut`

En la sesión 7 la columna `tamano` (pequeña, mediana, grande) se construyó con dos `np.where` anidados. `pd.cut` hace lo mismo en una línea, con los **cortes** y las **etiquetas** a la vista:

```
pd.cut(v["cantidad"], bins=[0, 2, 5, 10], labels=["pequeña", "mediana", "grande"])
                      └─ cortes: (0, 2]  (2, 5]  (5, 10] ─┘
```

- Los intervalos cierran **a la derecha**: `(0, 2]` incluye el 2. `right=False` los cierra a la izquierda.
- Un valor fuera de los cortes queda `NaN` (sin aviso). Elija el primer y el último corte con margen.
- Sin `labels`, la columna muestra los intervalos. El resultado es una columna **categórica ordenada**: `.cat.categories`, `sort_index()` respeta el orden.

`pd.qcut` corta por **cuantiles**: cuatro grupos con el mismo número de filas (cuartiles), diez (deciles)… Los cortes los decide el dato, no usted.

In [36]:
tamano_cut = pd.cut(v["cantidad"], bins=[0, 2, 5, 10], labels=["pequeña", "mediana", "grande"])
print("¿igual que la columna de la sesión 7?", (tamano_cut.astype(str) == v["tamano"]).all())
tamano_cut.value_counts().sort_index()

¿igual que la columna de la sesión 7? True


cantidad
pequeña    105
mediana     63
grande      32
Name: count, dtype: int64

In [37]:
# Sin etiquetas, cut muestra los intervalos; cada uno incluye su extremo derecho
pd.cut(v["cantidad"], bins=[0, 2, 5, 10]).value_counts().sort_index()

cantidad
(0, 2]     105
(2, 5]      63
(5, 10]     32
Name: count, dtype: int64

In [38]:
# qcut: cuatro grupos de ingreso con (casi) el mismo número de ventas. Los cortes salen de los cuantiles.
cuartil = pd.qcut(v["ingreso"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
print(cuartil.value_counts().sort_index().to_dict())
pd.qcut(v["ingreso"], 4).cat.categories

{'Q1': 52, 'Q2': 48, 'Q3': 53, 'Q4': 47}


IntervalIndex([(0.599, 6.4], (6.4, 15.5], (15.5, 50.0], (50.0, 270.0]], dtype='interval[float64, right]')

### Su turno 2.1 · 4 min
Cree `rango_precio` cortando `precio_unitario` en tres rangos: económico (hasta 5), medio (más de 5 hasta 20) y premium (más de 20 hasta 50). Cuente las ventas de cada rango.

In [39]:
# TODO: reemplace los ... por su código
rango_precio = pd.cut(v["precio_unitario"], bins=[0, 5, 20, 50], labels=["económico", "medio", "premium"])
rango_precio.value_counts().sort_index()

precio_unitario
económico    78
medio        89
premium      33
Name: count, dtype: int64

In [40]:
assert rango_precio.isna().sum() == 0, "hay precios fuera de los cortes: revise el primer y el último corte"
assert rango_precio.value_counts().to_dict() == {'medio': 89, 'económico': 78, 'premium': 33}, "revise los cortes (los intervalos cierran a la derecha)"
print("Correcto: económico", 78, "· medio", 89, "· premium", 33, ".")

Correcto: económico 78 · medio 89 · premium 33 .


## 3. Variables indicadoras: `get_dummies`

Los algoritmos de la Unidad 4 (distancias, *clustering*) solo entienden números. Una columna categórica se convierte en **una columna 0/1 por categoría**: la fila vale 1 en la columna de su categoría y 0 en las demás. Son las variables indicadoras (*dummies* o *one-hot*).

- `pd.get_dummies(serie, dtype=int)`: una tabla con tantas columnas como categorías. Sin `dtype=int` las columnas son `True`/`False` (pandas 2 y 3).
- `prefix="cat"` nombra las columnas `cat_Alimentos`, `cat_Hogar`…
- `drop_first=True` quita una columna (redundante: se deduce de las otras). Se usa en modelos; en la Unidad 4 no hace falta.

Hoy basta con saber construirlas y leerlas; se retoman en la Unidad 4.

In [41]:
pd.get_dummies(v["categoria"], prefix="cat", dtype=int).head()

,cat_Alimentos,cat_Electrónica,cat_Hogar,cat_Papelería
0,0,0,1,0
1,0,1,0,0
2,1,0,0,0
3,0,0,0,1
4,0,1,0,0


In [43]:
# Varias columnas a la vez: una tabla ancha, lista para la Unidad 4. Para pegarla a v: pd.concat([v, dummies], axis=1)
pd.get_dummies(v[["categoria", "metodo_pago"]], dtype=int).sum()

categoria_Alimentos          67
categoria_Electrónica        64
categoria_Hogar              36
categoria_Papelería          33
metodo_pago_desconocido       9
metodo_pago_efectivo         91
metodo_pago_tarjeta          73
metodo_pago_transferencia    27
dtype: int64

### Su turno 3.1 · 3 min
Construya `ind_tamano`, las variables indicadoras de `tamano` con valores 0/1 (sin prefijo). ¿Cuántas columnas tiene? ¿Cuánto suma la columna `pequeña`?

In [49]:
# TODO: reemplace los ... por su código
ind_tamano = pd.get_dummies(v["tamano"], dtype=int)
print(ind_tamano.shape)
ind_tamano.sum()

(200, 3)


grande      32
mediana     63
pequeña    105
dtype: int64

In [50]:
assert list(ind_tamano.shape) == [200, 3], "una fila por venta y una columna por categoría"
assert ind_tamano.dtypes.eq(int).all() or str(ind_tamano.dtypes.iloc[0]).startswith("int"), "use dtype=int para tener 0/1 en vez de True/False"
assert ind_tamano["pequeña"].sum() == 105
print("Correcto: cada fila suma 1 ->", (ind_tamano.sum(axis=1) == 1).all())

Correcto: cada fila suma 1 -> True


## 4. Unir tablas por una clave: `merge`

El costo de cada producto no está en las ventas: está en la hoja `productos` de `tienda.xlsx`. Las dos tablas comparten una columna, `producto`: es la **clave**. `merge` pega a cada venta la fila del producto que le corresponde.

```
v.merge(productos, on="producto", how="left")
        └ otra tabla ┘ └─ clave ─┘ └ qué filas conservar ┘
```

| `how=` | Conserva… | Si una venta no encuentra su producto |
| :-- | :-- | :-- |
| `"inner"` (por defecto) | solo las claves que están en **ambas** tablas | la venta **desaparece** |
| `"left"` | **todas** las filas de la izquierda | la venta queda, con `NaN` en las columnas nuevas |
| `"right"` | todas las de la derecha | — |
| `"outer"` | todas las de las dos | `NaN` donde falte |

Para enriquecer una tabla con un catálogo, `how="left"` es casi siempre lo correcto: no pierde ventas y deja ver (en los `NaN`) qué claves no se encontraron.

In [54]:
productos = pd.read_excel("datos/tienda.xlsx", sheet_name="productos")
locales = pd.read_excel("datos/tienda.xlsx", sheet_name="locales")
print(productos.shape, locales.shape)
productos

(10, 4) (5, 4)


,producto,categoria,precio_unitario,costo_unitario
0,Sartén,Hogar,18.0,11.50
1,Licuadora,Hogar,45.0,31.00
2,Mouse,Electrónica,15.0,9.80
3,Cargador,Electrónica,12.5,7.50
4,Audífonos,Electrónica,25.0,16.00
5,Galletas,Alimentos,1.8,1.10
6,Café,Alimentos,6.5,4.40
7,Chocolate,Alimentos,3.2,2.10
8,Cuaderno,Papelería,2.5,1.60
9,Bolígrafo,Papelería,0.6,0.35


In [55]:
# Solo las columnas que hacen falta: productos ya trae categoria y precio_unitario, y v también.
# Si se pegan dos columnas con el mismo nombre, merge las renombra _x y _y (evítelo eligiendo columnas).
vm = v.merge(productos[["producto", "costo_unitario"]], on="producto", how="left")
vm["margen"] = (vm["precio_unitario"] - vm["costo_unitario"]) * vm["cantidad"]
print(vm.shape, "| costo faltante:", vm["costo_unitario"].isna().sum(), "| margen total:", round(vm["margen"].sum(), 2))
vm.head()

(200, 12) | costo faltante: 0 | margen total: 2598.75


,fecha,producto,categoria,cantidad,precio_unitario,ciudad,metodo_pago,tamano,ingreso,region,costo_unitario,margen
0,2026-01-02,Sartén,Hogar,1,18.0,Quito,tarjeta,pequeña,18.0,Sierra,11.5,6.5
1,2026-01-02,Mouse,Electrónica,1,15.0,Machala,efectivo,pequeña,15.0,Costa,9.8,5.2
2,2026-01-03,Galletas,Alimentos,2,1.8,Quito,tarjeta,pequeña,3.6,Sierra,1.1,1.4
3,2026-01-03,Cuaderno,Papelería,2,2.5,Guayaquil,transferencia,pequeña,5.0,Costa,1.6,1.8
4,2026-01-04,Cargador,Electrónica,3,12.5,Guayaquil,tarjeta,mediana,37.5,Costa,7.5,15.0


In [56]:
# Comprobaciones que evitan sorpresas:
#   validate="many_to_one": muchas ventas por producto, pero UN solo producto por clave en el catálogo (si no, error).
#   indicator=True: columna _merge que dice de dónde salió cada fila.
vl = v.merge(locales, on="ciudad", how="left", validate="many_to_one", indicator=True)
vl["_merge"].value_counts()

_merge
both          194
left_only       6
right_only      0
Name: count, dtype: int64

In [ ]:
# Las 6 ventas con ciudad 'Sin dato' no tienen local: con how="left" quedan (con NaN); con how="inner" se pierden
print("left :", v.merge(locales, on="ciudad", how="left").shape)
print("inner:", v.merge(locales, on="ciudad", how="inner").shape)S

left : (200, 13)
inner: (194, 13)


Si las claves se llaman distinto en cada tabla: `left_on="ciudad", right_on="nombre_local"`. Si la clave está en el índice: `left_index=True` / `right_index=True`.

### Su turno 4.1 · 6 min
Pegue a las ventas la tabla `locales` (conservando **todas** las ventas) y calcule, por ciudad, el **ingreso por empleado**: ingreso total de la ciudad dividido por su número de empleados, redondeado a 1 decimal. Guarde la tabla agregada en `por_ciudad` (una fila por ciudad, columnas `ingreso`, `empleados`, `ingreso_por_empleado`). ¿Qué ciudad rinde más por empleado?

In [59]:
# TODO: reemplace los ... por su código
vl = v.merge(locales, on="ciudad", how="left")
por_ciudad = vl.groupby("ciudad").agg(ingreso=("ingreso", "sum"), empleados=("empleados", "first"))
por_ciudad["ingreso_por_empleado"] = (por_ciudad["ingreso"] / por_ciudad["empleados"]).round(1)
por_ciudad.sort_values("ingreso_por_empleado", ascending=False)

,ingreso,empleados,ingreso_por_empleado
ciudad,,,
Quito,2512.6,12.0,209.4
Guayaquil,2745.7,14.0,196.1
Manta,694.0,5.0,138.8
Machala,532.5,4.0,133.1
Cuenca,865.3,7.0,123.6
Sin dato,74.7,NaN,NaN


In [60]:
assert len(vl) == len(v), "se perdieron ventas: use how='left'"
assert vl["empleados"].isna().sum() == 6, "las ventas 'Sin dato' deben quedar con NaN en empleados"
assert np.isclose(por_ciudad.loc["Guayaquil", "ingreso_por_empleado"], 196.1), "ingreso total de la ciudad / empleados, redondeado a 1 decimal"
print("Correcto. La ciudad con más ingreso por empleado es", por_ciudad["ingreso_por_empleado"].idxmax(), ".")

Correcto. La ciudad con más ingreso por empleado es Quito .


In [61]:
# Para continuar con la misma tabla: v queda con costo_unitario y margen (ejecute siempre esta celda)
v = v.merge(productos[["producto", "costo_unitario"]], on="producto", how="left")
v["margen"] = (v["precio_unitario"] - v["costo_unitario"]) * v["cantidad"]
v.shape

(200, 12)

## 5. Apilar tablas: `concat`

Las ventas de abril llegaron en **otro archivo**, con el mismo tipo de filas. Apilar tablas es `pd.concat([tabla1, tabla2])`:

- Alinea las columnas **por nombre**: una columna que falte en una de las tablas queda `NaN` en esas filas.
- Conserva los índices originales (dos filas con etiqueta 0): `ignore_index=True` renumera.
- `axis=1` pega **de lado** (columnas), alineando por índice; es lo que hace falta para añadir las *dummies*.

Antes de apilar, las dos tablas deben tener los **mismos nombres y tipos** en las columnas comunes. `ventas_abril.csv` está como llegó (sesión 7, «Su turno 8.1»): separador `;`, decimal `,`, codificación latin-1 y fechas `dd/mm/aaaa`. Si se apila sin limpiar, `fecha` queda como texto y `precio_unitario` también: no se puede sumar.

In [62]:
# Limpieza mínima de abril, como en la sesión 7 (esto NO es nuevo)
abril = pd.read_csv("datos/ventas_abril.csv", sep=";", decimal=",", encoding="latin-1")
abril["fecha"] = pd.to_datetime(abril["fecha"], format="%d/%m/%Y")
abril["ingreso"] = abril["cantidad"] * abril["precio_unitario"]
print(abril.shape, "| ingreso de abril:", round(abril["ingreso"].sum(), 1))
print("columnas que v tiene y abril no:", sorted(set(v.columns) - set(abril.columns)))
abril.dtypes

(60, 7) | ingreso de abril: 2070.4
columnas que v tiene y abril no: ['costo_unitario', 'margen', 'metodo_pago', 'region', 'tamano']


fecha              datetime64[us]
producto                      str
categoria                     str
cantidad                    int64
precio_unitario           float64
ciudad                        str
ingreso                   float64
dtype: object

In [63]:
# Apilar sin más: 260 filas; las columnas que abril no trae quedan NaN en sus 60 filas
junto = pd.concat([v, abril], ignore_index=True)
print(junto.shape)
junto.isna().sum()[lambda s: s > 0]

(260, 12)


metodo_pago       60
tamano            60
region            60
costo_unitario    60
margen            60
dtype: int64

Los `NaN` dicen exactamente qué hay que completar: `costo_unitario` y `margen` se obtienen con el mismo `merge` de la sección 4; `region` con el mismo `map`; `metodo_pago` no existe en abril (queda `"desconocido"`, la misma etiqueta de la sesión 7); y `tamano` se **recalcula para toda la tabla** con `cut` (una línea, en vez de repetir los `np.where`).

### Su turno 5.1 · 6 min
Construya `completo` con las 260 ventas de enero a abril:
1. Complete `abril` con `costo_unitario` y `margen` (`merge` con `productos`) y `region` (`map` + `fillna`).
2. Apile `v` y `abril` con índice renumerado.
3. Rellene `metodo_pago` con `"desconocido"` y recalcule `tamano` con `cut` para las 260 filas.

In [64]:
# TODO: reemplace los ... por su código
abril = abril.merge(productos[["producto", "costo_unitario"]], on="producto", how="left")
abril["margen"] = (abril["precio_unitario"] - abril["costo_unitario"]) * abril["cantidad"]
abril["region"] = abril["ciudad"].map(REGION).fillna("Sin dato")

completo = pd.concat([v, abril], ignore_index=True)
completo["metodo_pago"] = completo["metodo_pago"].fillna("desconocido")
completo["tamano"] = pd.cut(completo["cantidad"], bins=[0, 2, 5, 10], labels=["pequeña", "mediana", "grande"])
print(completo.shape, "| ingreso total:", round(completo["ingreso"].sum(), 1))
completo.isna().sum().sum()

(260, 12) | ingreso total: 9495.2


np.int64(0)

In [65]:
assert len(completo) == 260, f"deben ser 260 filas (200 + 60) y hay {len(completo)}"
assert completo.index.max() == 259, "renumere el índice con ignore_index=True"
assert np.isclose(completo["ingreso"].sum(), 9495.2) and np.isclose(completo["margen"].sum(), 3327.3), "revise el merge de abril con productos"
assert completo.isna().sum().sum() == 0, "quedan NaN: revise metodo_pago, region, costo_unitario y tamano"
assert completo["tamano"].value_counts().to_dict() == {'pequeña': 138, 'mediana': 81, 'grande': 41}, "recalcule tamano con cut sobre las 260 filas"
print("Correcto: 260 ventas de enero a abril, ingreso", round(completo["ingreso"].sum(), 1), ", margen", round(completo["margen"].sum(), 2), ".")

Correcto: 260 ventas de enero a abril, ingreso 9495.2 , margen 3327.3 .


In [66]:
# Para continuar con la misma tabla (ejecute siempre esta celda): construye `completo` de forma canónica
abril = pd.read_csv("datos/ventas_abril.csv", sep=";", decimal=",", encoding="latin-1")
abril["fecha"] = pd.to_datetime(abril["fecha"], format="%d/%m/%Y")
abril["ingreso"] = abril["cantidad"] * abril["precio_unitario"]
abril = abril.merge(productos[["producto", "costo_unitario"]], on="producto", how="left")
abril["margen"] = (abril["precio_unitario"] - abril["costo_unitario"]) * abril["cantidad"]
abril["region"] = abril["ciudad"].map(REGION).fillna("Sin dato")
completo = pd.concat([v, abril], ignore_index=True)
completo["metodo_pago"] = completo["metodo_pago"].fillna("desconocido")
completo["tamano"] = pd.cut(completo["cantidad"], bins=[0, 2, 5, 10], labels=["pequeña", "mediana", "grande"])
# desde aquí trabajamos con las 260 ventas ordenadas por fecha y con la columna mes
completo = completo.sort_values("fecha", ignore_index=True)
completo["mes"] = completo["fecha"].dt.month
completo.groupby("mes")["ingreso"].sum().round(1)

mes
1    1835.8
2    2400.1
3    3188.9
4    2070.4
Name: ingreso, dtype: float64

## 6. Formato largo y formato ancho

La misma información cabe en dos formas:

```
LARGO (una fila por observación)          ANCHO (una fila por ciudad, una columna por mes)
 ciudad     mes   ingreso                   mes          1       2       3       4
 Cuenca      1     195.0                    ciudad
 Cuenca      2     355.7          <--->     Cuenca    195.0   355.7   314.6   556.4
 Guayaquil   1     938.9                    Guayaquil 938.9   932.7   874.1   662.7
 ...
```

- **Ancho** es lo que lee una persona; es lo que devuelven `pivot_table` (sesión 8) y `unstack`.
- **Largo** es lo que prefieren `groupby`, los gráficos y los archivos de datos abiertos. Para volver de ancho a largo: `melt`.

| De… a… | Código |
| :-- | :-- |
| largo → ancho (resumiendo) | `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)` |
| largo → ancho (sin resumir: una fila por combinación) | `df.pivot(index=..., columns=..., values=...)` |
| ancho → largo | `ancho.reset_index().melt(id_vars=..., var_name=..., value_name=...)` |
| nivel de columnas ↔ nivel de índice | `stack()` / `unstack()` |

In [67]:
ancho = completo.pivot_table(index="ciudad", columns="mes", values="ingreso", aggfunc="sum", fill_value=0).round(1)
ancho

mes,1,2,3,4
ciudad,,,,
Cuenca,195.0,355.7,314.6,556.4
Guayaquil,938.9,932.7,874.1,662.7
Machala,56.0,209.0,267.5,177.1
Manta,59.6,99.6,534.8,69.1
Quito,555.1,759.6,1197.9,605.1
Sin dato,31.2,43.5,0.0,0.0


In [68]:
# De ancho a largo: id_vars se repite; las demás columnas se apilan en dos columnas (variable, valor)
largo = ancho.reset_index().melt(id_vars="ciudad", var_name="mes", value_name="ingreso")
print(largo.shape)
largo.head(8)

(24, 3)


,ciudad,mes,ingreso
0,Cuenca,1,195.0
1,Guayaquil,1,938.9
2,Machala,1,56.0
3,Manta,1,59.6
4,Quito,1,555.1
5,Sin dato,1,31.2
6,Cuenca,2,355.7
7,Guayaquil,2,932.7


In [69]:
# El camino de vuelta: pivot (sin aggfunc) porque ya hay UNA fila por combinación ciudad-mes
largo.pivot(index="ciudad", columns="mes", values="ingreso").equals(ancho)

True

### Su turno 6.1 · 4 min
Construya `unidades_ancho`: unidades vendidas (`cantidad`, suma) por producto (filas) y mes (columnas), con ceros donde no hubo ventas. Después páselo a formato largo en `unidades_largo`, con columnas `producto`, `mes` y `unidades`. ¿Cuántas filas tiene el formato largo?

In [70]:
# TODO: reemplace los ... por su código
unidades_ancho = completo.pivot_table(index="producto", columns="mes", values="cantidad", aggfunc="sum", fill_value=0)
unidades_largo = unidades_ancho.reset_index().melt(id_vars="producto", var_name="mes", value_name="unidades")
print(unidades_largo.shape)
unidades_largo.head()

(40, 3)


,producto,mes,unidades
0,Audífonos,1,17
1,Bolígrafo,1,17
2,Café,1,18
3,Cargador,1,7
4,Chocolate,1,22


In [71]:
assert list(unidades_largo.columns) == ["producto", "mes", "unidades"], "nombre las columnas con var_name y value_name"
assert len(unidades_largo) == 40, "una fila por combinación producto-mes"
assert unidades_largo["unidades"].sum() == 817, "la suma del formato largo debe ser el total de unidades"
print("Correcto:", unidades_ancho.shape[0], "productos x", unidades_ancho.shape[1], "meses =", len(unidades_largo), "filas en formato largo.")

Correcto: 10 productos x 4 meses = 40 filas en formato largo.


## 7. Guardar el resultado: `datos/ventas_completo.csv`

La tabla de la sesión 10 es esta: 260 ventas de enero a abril con región, costo y margen. Se guarda **sin el índice** y se vuelve a leer para comprobar que quedó bien. Un CSV no guarda el tipo `category` ni las fechas como fechas: al leerlo, `parse_dates=["fecha"]` y, si hace falta, `astype("category")`.

In [ ]:
columnas = ["fecha", "producto", "categoria", "cantidad", "precio_unitario", "costo_unitario", "ciudad", "region",
            "metodo_pago", "tamano", "ingreso", "margen", "mes"]
completo[columnas].to_csv("datos/ventas_completo.csv", index=False)

prueba = pd.read_csv("datos/ventas_completo.csv", parse_dates=["fecha"])
print(prueba.shape, "| ingreso:", round(prueba["ingreso"].sum(), 1), "| margen:", round(prueba["margen"].sum(), 2))
prueba.dtypes

## 8. Errores frecuentes

| Síntoma | Causa | Solución |
| :-- | :-- | :-- |
| después de `merge` hay **más filas** que antes | la clave se repite en la tabla de la derecha (muchos a muchos) | `validate="many_to_one"` avisa; quite duplicados del catálogo |
| después de `merge` hay **menos filas** | `how="inner"` descartó claves sin pareja | `how="left"`; mire `indicator=True` |
| columnas `precio_x` y `precio_y` | las dos tablas traían la misma columna | seleccione solo las columnas nuevas antes del `merge` |
| `KeyError` en `merge` | la clave se llama distinto en cada tabla | `left_on=`, `right_on=` |
| `NaN` en una columna después de `map` | valores que no están en el diccionario | `fillna(...)` o complete el diccionario |
| `NaN` después de `cut` | valores fuera del primer o del último corte | amplíe los cortes; revise `right=` |
| dos filas con índice 0 después de `concat` | los índices originales se conservan | `ignore_index=True` |
| `concat` llena de `NaN` | las columnas tienen nombres distintos en cada tabla | renombre antes (`rename(columns=...)`) |
| las *dummies* salen `True`/`False` | comportamiento de pandas 2 y 3 | `dtype=int` |

## 9. Hacia el proyecto

Casi todos los conjuntos de datos del curso necesitan algo de hoy antes de la exploración:

| Situación en su proyecto | Herramienta |
| :-- | :-- |
| un archivo por año, por mes o por provincia (p. ej., la matrícula del MINEDUC por período lectivo) | `concat` (limpiar cada uno igual antes) |
| una tabla de códigos aparte (código de provincia → nombre, código de producto → categoría) | `merge` con `how="left"` o `map` con un diccionario |
| una variable numérica que la pregunta trata por grupos (edad → grupo de edad, horas → nivel de uso) | `cut` (cortes con sentido) o `qcut` (cuantiles) |
| muchas categorías que la pregunta agrupa (país → región, 24 provincias → 4 regiones) | `map` |
| una tabla ancha (una columna por año) que hay que graficar o agrupar | `melt` |

### Su turno 9.1 · en casa
Con el conjunto de datos de su proyecto (la tabla limpia de la Tarea 1): (a) cree **una** columna nueva con `map`, `cut` o `qcut` que haga falta para alguna de sus preguntas de negocio; (b) si el grupo tiene **más de un archivo o tabla**, júntelos con `concat` o `merge` y compruebe el número de filas antes y después; si tiene uno solo, pase una `pivot_table` de su tabla a formato largo con `melt`; (c) escriba en una celda de texto qué cambió y por qué. Traiga el notebook a la sesión 10.

In [ ]:
# Plantilla para su proyecto (complete con sus columnas)
# datos = pd.read_csv("datos/mi_dataset_limpio.csv")
# datos["grupo"] = pd.cut(datos["columna_numerica"], bins=[...], labels=[...])
# datos = datos.merge(catalogo[["codigo", "nombre"]], on="codigo", how="left", validate="many_to_one")
# todo = pd.concat([anio_2023, anio_2024], ignore_index=True)

**Antes de la próxima sesión (S10):** traiga `datos/ventas_completo.csv` (lo escribió la sección 7) y el notebook de «Su turno 9.1». En la S10 se describe cada variable por separado (descriptivos, tablas de frecuencia, histogramas y diagramas de caja) y se empieza a graficar con matplotlib. Lecturas: McKinney cap. 8 (8.2 «Combinar y mezclar» y 8.3 «Remodelar y pivotar»); VanderPlas cap. 3, «Combining Datasets: Concat and Append» y «Combining Datasets: Merge and Join».